# Laboratorio 5 (Soluzioni) — AI moderna per il suono: embedding, VAE e diffusione

Soluzioni complete dei quattro esercizi paradigmatici del Capitolo 5 con `audiolab`.

## Configurazione

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import soundfile as sf
from IPython.display import Audio, display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "lab-2026/audiolab").is_dir())
sys.path.insert(0, str(ROOT / "lab-2026"))
import audiolab as al

plt.rcParams.update({"figure.figsize": (9, 3.8), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
print("Radice del progetto:", ROOT)


Radice del progetto: /Users/grossi/didattica/InfoMM-local


## Esercizio 1 — Spazio condiviso audio-testo e matrice di similarità (CLAP)

Il modello preaddestrato **CLAP** (*Contrastive Language-Audio Pretraining*, `al.CLAP_MODEL_ID`) proietta suoni e frasi nello stesso spazio vettoriale di 512 dimensioni. Carichiamo i vettori reali precalcolati in `lab-2026/materiali/05-clap` (generabili con `al.prepare_clap_embeddings()`), normalizziamo ogni riga a norma unitaria e calcoliamo la matrice di similarità coseno $S = A T^\top$ fra i tre clip audio e le quattro descrizioni candidate `al.PHRASES`.

### Esercizio 1 — Classificazione zero-shot e rimozione della descrizione corretta

1. Visualizza la matrice `sim` ($3\times 4$) e verifica che sulla diagonale principale compaiano i punteggi più alti per parlato, chitarra e abbaio.
2. Simula il caso in cui la descrizione corretta dell'abbaio (`"The sound of a dog barking."`, colonna 2) **non sia presente** nell'elenco delle candidate: tieni solo le colonne `[0, 1, 3]` e verifica quale descrizione viene scelta da $\arg\max$ per l'abbaio (`sim[2, [0, 1, 3]]`) e con quale punteggio.
3. Trasforma i punteggi della riga dell'abbaio in pesi di somma 1 tramite una softmax con temperatura $\tau = 0{,}07$ ($p_j = \exp(s_j/\tau)/\sum_k \exp(s_k/\tau)$) prima e dopo la rimozione della colonna 2: perché il peso massimo sale anche quando la risposta corretta non c'è?

In [2]:
base_clap = ROOT / "lab-2026/materiali/05-clap"
meta_clap = json.loads((base_clap / "provenienza.json").read_text())
with np.load(base_clap / "embedding.npz") as archivio:
    audio_emb = archivio["audio"]
    testo_emb = archivio["testo"]

audio_norm = audio_emb / np.linalg.norm(audio_emb, axis=1, keepdims=True)
testo_norm = testo_emb / np.linalg.norm(testo_emb, axis=1, keepdims=True)
sim = audio_norm @ testo_norm.T

print("Modello:", meta_clap["model_id"], "| dimensione embedding:", audio_emb.shape[1])
for i, percorso in enumerate(meta_clap["files"]):
    j_max = int(np.argmax(sim[i]))
    print(f"  {Path(percorso).name:22s} -> {meta_clap['frasi'][j_max]} (coseno = {sim[i, j_max]:.3f})")

tau = 0.07
indici_ridotti = [0, 1, 3]
frasi_ridotte = [meta_clap["frasi"][j] for j in indici_ridotti]
sim_abbaio_rid = sim[2, indici_ridotti]
j_vincitore_rid = int(np.argmax(sim_abbaio_rid))

pesi_4 = np.exp(sim[2] / tau) / np.sum(np.exp(sim[2] / tau))
pesi_3 = np.exp(sim_abbaio_rid / tau) / np.sum(np.exp(sim_abbaio_rid / tau))

print("Abbaio (4 candidate) -> vince:", meta_clap["frasi"][int(np.argmax(sim[2]))],
      f"| coseno = {sim[2].max():.3f} | peso softmax = {pesi_4.max():.3f}")
print("Abbaio (senza 'dog barking') -> vince:", frasi_ridotte[j_vincitore_rid],
      f"| coseno = {sim_abbaio_rid.max():.3f} | peso softmax = {pesi_3.max():.3f}")

Modello: laion/clap-htsat-unfused | dimensione embedding: 512
  speech1.wav            -> The sound of a person speaking. (coseno = 0.396)
  Guitar_C4.wav          -> The sound of a guitar playing a note. (coseno = 0.506)
  100032-3-0-0.mp3       -> The sound of a dog barking. (coseno = 0.685)
Abbaio (4 candidate) -> vince: The sound of a dog barking. | coseno = 0.685 | peso softmax = 0.995
Abbaio (senza 'dog barking') -> vince: The sound of a person speaking. | coseno = 0.277 | peso softmax = 0.588


## Esercizio 2 — Ricostruzione neurale con un VAE audio (RAVE)

Il modello **RAVE** (IRCAM) è un *autoencoder variazionale* (VAE) addestrato su voci parlate e cantate a $48\,\text{kHz}$: comprime ogni blocco di $2048$ campioni ($\approx 42{,}7\,\text{ms}$) in un vettore latente continuo di **11 numeri** e da lì ricostruisce la forma d'onda. Carichiamo i materiali con `al.load_materials()`, ascoltiamo `voce1` originale e ricostruita con `al.compare_audio()` e misuriamo la fedeltà con `al.reconstruction_metrics()`.

### Esercizio 2 — Confronto oggettivo e uditivo dentro dominio (`voce2`) e fuori dominio (`abbaio`)

1. Calcola con `al.reconstruction_metrics(riferimento, ricostruzione, sr=sr_vae)` le metriche di ricostruzione sulla seconda voce (`materiali_vae["voce2"]` vs `materiali_vae["ricostruite"][1]`) e sul suono fuori dominio (`materiali_vae["fuori_dominio"]["originale"]` vs `materiali_vae["fuori_dominio"]["ricostruito"]`, un abbaio passato nel modello addestrato solo su voci).
2. Ascolta la coppia fuori dominio con `al.compare_audio()` e confronta i valori di `inviluppo` (correlazione dell'inviluppo RMS) e `mel` (similarità coseno fra spettrogrammi log-Mel): perché la ricostruzione dell'abbaio assume un timbro quasi vocale?

In [3]:
materiali_vae = al.load_materials()
sr_vae = materiali_vae["sr"]
voce1 = materiali_vae["voce1"]
voce1_ric = materiali_vae["ricostruite"][0]

m_voce1 = al.reconstruction_metrics(voce1, voce1_ric, sr=sr_vae)
print("Metriche di ricostruzione su voce1:", {k: round(v, 3) for k, v in m_voce1.items()})
al.compare_audio([("Voce 1 originale", voce1[:3 * sr_vae]),
                  ("Voce 1 ricostruita (RAVE)", voce1_ric[:3 * sr_vae])], sr=sr_vae)

voce2 = materiali_vae["voce2"]
voce2_ric = materiali_vae["ricostruite"][1]
abbaio_orig = materiali_vae["fuori_dominio"]["originale"]
abbaio_ric = materiali_vae["fuori_dominio"]["ricostruito"]

m_voce2 = al.reconstruction_metrics(voce2, voce2_ric, sr=sr_vae)
m_fuori = al.reconstruction_metrics(abbaio_orig, abbaio_ric, sr=sr_vae)

print("Dentro dominio (voce2): ", {k: round(v, 3) for k, v in m_voce2.items()})
print("Fuori dominio (abbaio):", {k: round(v, 3) for k, v in m_fuori.items()})
al.compare_audio([("Abbaio originale", abbaio_orig),
                  ("Abbaio ricostruito dal VAE voce", abbaio_ric)], sr=sr_vae)

Metriche di ricostruzione su voce1: {'ritardo': 487, 'inviluppo': 0.951, 'mel': 0.924}
Guadagno comune per l'ascolto: 0.0 dB


**Voce 1 originale** — 3.00 s; picco 0.370; RMS 0.0867

**Voce 1 ricostruita (RAVE)** — 3.00 s; picco 0.599; RMS 0.0813

Dentro dominio (voce2):  {'ritardo': 513, 'inviluppo': 0.958, 'mel': 0.94}
Fuori dominio (abbaio): {'ritardo': 291, 'inviluppo': 0.92, 'mel': 0.802}
Guadagno comune per l'ascolto: -2.7 dB


**Abbaio originale** — 0.37 s; picco 0.918; RMS 0.1445

**Abbaio ricostruito dal VAE voce** — 0.37 s; picco 1.092; RMS 0.1365

## Esercizio 3 — Attraversare lo spazio latente: morphing tra due voci

Poiché lo spazio latente di un VAE è reso continuo e regolare durante l'addestramento, possiamo interpolare linearmente i latenti delle due voci:
$$\mathbf{z}_\alpha = (1 - \alpha)\,\mathbf{z}_1 + \alpha\,\mathbf{z}_2, \qquad \alpha \in [0, 1]$$
e decodificare ciascun punto intermedio. Osserviamo la dimensione dei latenti in `materiali_vae["latenti"]` e ascoltiamo i tre passi principali ($\alpha = 0$, $\alpha = 0{,}5$, $\alpha = 1$) con `al.compare_audio()`.

### Esercizio 3 — Traiettoria timbrica lungo il morphing con `al.mean_mfcc()`

1. Calcola il vettore timbrico medio (13 MFCC mediati nel tempo) delle due voci di riferimento `voce1` e `voce2` usando `al.mean_mfcc(y, sr=sr_vae)`.
2. Per ciascuno dei 9 segnali `(alpha, y_m)` in `catena_morph`, calcola `al.mean_mfcc(y_m, sr=sr_vae)` e misura la distanza euclidea (`np.linalg.norm`) dal timbro di `voce1` e da quello di `voce2`.
3. Disegna le due curve in funzione di $\alpha \in [0, 1]$ con `al.plot_curves()` e individua il passo $\alpha$ in cui il timbro smette di assomigliare più a `voce1` e diventa più vicino a `voce2`.

In [4]:
z1 = materiali_vae["latenti"]["voce1"]
z2 = materiali_vae["latenti"]["voce2"]
catena_morph = materiali_vae["morph"]  # lista di 9 coppie (alpha, segnale)

print(f"Forma del latente z1: {z1.shape} (11 coordinate x {z1.shape[1]} blocchi da 2048 campioni)")
al.compare_audio([
    (f"alpha = {catena_morph[0][0]:.2f} (Voce 1)", catena_morph[0][1][:3 * sr_vae]),
    (f"alpha = {catena_morph[4][0]:.2f} (Centro)", catena_morph[4][1][:3 * sr_vae]),
    (f"alpha = {catena_morph[8][0]:.2f} (Voce 2)", catena_morph[8][1][:3 * sr_vae]),
], sr=sr_vae)

mf1 = al.mean_mfcc(voce1, sr=sr_vae)
mf2 = al.mean_mfcc(voce2, sr=sr_vae)
alphas = np.array([a for a, _ in catena_morph])

dist_da_1 = np.array([np.linalg.norm(al.mean_mfcc(y_m, sr=sr_vae) - mf1) for _, y_m in catena_morph])
dist_da_2 = np.array([np.linalg.norm(al.mean_mfcc(y_m, sr=sr_vae) - mf2) for _, y_m in catena_morph])

idx_incrocio = int(np.argmin(np.abs(dist_da_1 - dist_da_2)))
print(f"Punto più vicino all'incrocio: alpha = {alphas[idx_incrocio]:.3f} "
      f"(d1 = {dist_da_1[idx_incrocio]:.1f}, d2 = {dist_da_2[idx_incrocio]:.1f})")

al.plot_curves(
    [("Distanza MFCC da Voce 1", alphas, dist_da_1),
     ("Distanza MFCC da Voce 2", alphas, dist_da_2)],
    title="Traiettoria timbrica lungo l'interpolazione latente RAVE",
    x_title="Parametro di morphing alpha",
    y_title="Distanza euclidea MFCC",
    mode="lines+markers",
)

Forma del latente z1: (11, 118) (11 coordinate x 118 blocchi da 2048 campioni)
Guadagno comune per l'ascolto: 0.0 dB


**alpha = 0.00 (Voce 1)** — 3.00 s; picco 0.733; RMS 0.0810

**alpha = 0.50 (Centro)** — 3.00 s; picco 0.365; RMS 0.0434

**alpha = 1.00 (Voce 2)** — 3.00 s; picco 0.726; RMS 0.0736

Punto più vicino all'incrocio: alpha = 0.125 (d1 = 40.1, d2 = 41.4)


## Esercizio 4 — Generazione testo-audio per diffusione latente (AudioLDM)

In **AudioLDM** un processo di diffusione raffina in $N=25$ passi una rappresentazione latente guidata dall'embedding del prompt testuale; il decoder e un vocoder neurale sintetizzano poi l'audio a $16\,\text{kHz}$. Carichiamo con `al.load_samples()` i tre esperimenti controllati (`A`: seed 42, stanza asciutta; `B`: seed 43, stanza asciutta; `C`: seed 42, stanza riverberante) e ascoltiamoli mantenendo una scala di ampiezza comune.

### Esercizio 4 — Verifica quantitativa e spettrogrammi dei tre clip generati (`A`, `B`, `C`)

1. Per ciascun clip `(nome, y, sr_g, meta_g)` in `clips_gen`, calcola il livello di picco in dBFS (`al.to_dbfs(np.max(np.abs(y)))`), il livello RMS in dBFS (`al.to_dbfs(al.rms(y))`) e i descrittori spettrali (`al.spectral_features(y, sr_g)`).
2. Disegna lo spettrogramma di `Clip A` (stanza asciutta, seed 42) e di `Clip C` (stanza riverberante, seed 42) con `al.plot_spectrogram(y, sr_g, mel=True)`.
3. Commenta i due confronti sperimentali:
   - **Effetto del seed (`A` vs `B`, stesso prompt)**: perché non bisogna mai valutare o scegliere un modello generativo su un singolo seed?
   - **Effetto del prompt (`A` vs `C`, stesso seed)**: il vincolo `"A single short dog bark"` è stato rispettato in entrambi i clip?

In [5]:
clips_gen = al.load_samples()
picco_comune = max(float(np.max(np.abs(y))) for _, y, _, _ in clips_gen)
guadagno_comune = min(1.0, 0.8 / max(picco_comune, 1e-12))

for nome, y, sr_g, meta_g in clips_gen:
    print(f"Clip {nome} | seed={meta_g['seed']} | passi={meta_g['steps']} | "
          f"guidance={meta_g['guidance_scale']} | tempo={meta_g['inference_seconds']:.1f} s")
    print(f"       prompt: '{meta_g['prompt']}'")

al.compare_audio([(f"Clip {nome} (seed {m['seed']})", y)
                  for nome, y, _, m in clips_gen], sr=clips_gen[0][2])

for nome, y, sr_g, meta_g in clips_gen:
    picco_db = al.to_dbfs(np.max(np.abs(y)))
    rms_db = al.to_dbfs(al.rms(y))
    centroide, banda, flatness, rolloff = al.spectral_features(y, sr_g)
    print(f"Clip {nome} (seed {meta_g['seed']}) -> picco: {picco_db:6.1f} dBFS | "
          f"RMS: {rms_db:6.1f} dBFS | centroide: {centroide:6.1f} Hz | "
          f"rolloff: {rolloff:6.1f} Hz")

al.plot_spectrogram(clips_gen[0][1], clips_gen[0][2], mel=True,
                    title="Clip A — seed 42, 'quiet dry room'")
al.plot_spectrogram(clips_gen[2][1], clips_gen[2][2], mel=True,
                    title="Clip C — seed 42, 'quiet reverberant room'")

Clip A | seed=42 | passi=25 | guidance=2.5 | tempo=17.3 s
       prompt: 'A single short dog bark in a quiet dry room.'
Clip B | seed=43 | passi=25 | guidance=2.5 | tempo=3.0 s
       prompt: 'A single short dog bark in a quiet dry room.'
Clip C | seed=42 | passi=25 | guidance=2.5 | tempo=3.3 s
       prompt: 'A single short dog bark in a quiet reverberant room.'
Guadagno comune per l'ascolto: -0.7 dB


**Clip A (seed 42)** — 5.00 s; picco 0.868; RMS 0.0441

**Clip B (seed 43)** — 5.00 s; picco 0.000; RMS 0.0002

**Clip C (seed 42)** — 5.00 s; picco 0.467; RMS 0.0288

Clip A (seed 42) -> picco:   -1.2 dBFS | RMS:  -27.1 dBFS | centroide: 1064.2 Hz | rolloff: 1450.0 Hz
Clip B (seed 43) -> picco:  -73.5 dBFS | RMS:  -74.7 dBFS | centroide:  441.4 Hz | rolloff:   72.6 Hz
Clip C (seed 42) -> picco:   -6.6 dBFS | RMS:  -30.8 dBFS | centroide: 1095.5 Hz | rolloff: 1521.2 Hz
